# Spectral Clustering

## Learning Objectives

1. **Construct** the graph Laplacian L = D − A and its normalized form
2. **Explain** why the Fiedler vector partitions a graph
3. **Implement** spectral clustering using the k smallest eigenvectors
4. **Connect** spectral clustering to normalized cut minimization

## The Graph Laplacian

Given graph $G$ with adjacency matrix $A$ and degree matrix $D$ (diagonal, $D_{ii} = d_i$):

$$L = D - A$$

**Key property:** for any vector $\mathbf{x}$:
$$\mathbf{x}^T L \mathbf{x} = \sum_{(i,j) \in E} (x_i - x_j)^2 \geq 0$$

So $L$ is positive semi-definite. Its eigenvalues are $0 = \lambda_1 \leq \lambda_2 \leq \ldots \leq \lambda_n$.

**Number of zero eigenvalues = number of connected components.**

The **Fiedler value** $\lambda_2$ (algebraic connectivity) measures how well-connected the graph is.

In [1]:
import numpy as np

def graph_laplacian(adj):
    nodes = sorted(adj.keys())
    n = len(nodes); idx = {v:i for i,v in enumerate(nodes)}
    A = np.zeros((n,n))
    for u in adj:
        for v in adj[u]:
            A[idx[u],idx[v]] = 1
    D = np.diag(A.sum(axis=1))
    L = D - A
    return L, nodes

from collections import defaultdict
# Two-community graph with a bridge
adj = defaultdict(set)
c1=[0,1,2,3,4]; c2=[5,6,7,8,9]
for grp in [c1,c2]:
    for i in grp:
        for j in grp:
            if i<j: adj[i].add(j); adj[j].add(i)
adj[2].add(5); adj[5].add(2)   # single bridge

L, nodes = graph_laplacian(adj)
vals, vecs = np.linalg.eigh(L)
print("Smallest 4 eigenvalues:", np.round(vals[:4],4))
print("Fiedler value (λ₂):", round(vals[1],4))

Smallest 4 eigenvalues: [-0.      0.2984  5.      5.    ]
Fiedler value (λ₂): 0.2984


## The Fiedler Vector

The **Fiedler vector** is the eigenvector corresponding to $\lambda_2$ (the second smallest eigenvalue).

**Partition by sign:**
$$S = \{i : f_i < 0\}, \quad \bar{S} = \{i : f_i \geq 0\}$$

This partition minimizes the **ratio cut**:
$$\text{RatioCut}(S,\bar{S}) = \frac{\text{cut}(S,\bar{S})}{|S|} + \frac{\text{cut}(S,\bar{S})}{|\bar{S}|}$$

**Intuition:** nodes on the same side of the cut have similar Fiedler vector values; the cut edges cause discontinuities.

In [2]:
fiedler = vecs[:, 1]   # second eigenvector
print("Fiedler vector values (by node):")
for i, node in enumerate(nodes):
    comm = "C1" if fiedler[i] < 0 else "C2"
    print(f"  node {node}: {fiedler[i]:+.4f}  → {comm}")

predicted = {node: (0 if fiedler[i] < 0 else 1) for i, node in enumerate(nodes)}
true_part  = {n:0 for n in c1}; true_part.update({n:1 for n in c2})
correct = sum(predicted[n]==true_part[n] for n in nodes)
print(f"""
Partition accuracy: {correct}/{len(nodes)}""")

Fiedler vector values (by node):
  node 0: -0.3336  → C1
  node 1: -0.3336  → C1
  node 2: -0.2341  → C1
  node 3: -0.3336  → C1
  node 4: -0.3336  → C1
  node 5: +0.2341  → C2
  node 6: +0.3336  → C2
  node 7: +0.3336  → C2
  node 8: +0.3336  → C2
  node 9: +0.3336  → C2

Partition accuracy: 10/10


## Normalized Laplacian

The **normalized Laplacian** $L_{\text{sym}} = D^{-1/2} L D^{-1/2}$ accounts for degree heterogeneity.

$$L_{\text{sym}} = I - D^{-1/2} A D^{-1/2}$$

Eigenvalues of $L_{\text{sym}}$ are in $[0,2]$. The Fiedler vector of $L_{\text{sym}}$ minimizes the **normalized cut**:

$$\text{NCut}(S,\bar{S}) = \frac{\text{cut}(S,\bar{S})}{\text{vol}(S)} + \frac{\text{cut}(S,\bar{S})}{\text{vol}(\bar{S})}$$

where $\text{vol}(S) = \sum_{u\in S} d_u$. NCut is preferred for graphs with heterogeneous degrees.

## k-Way Spectral Clustering

For $k$ communities, use the $k$ smallest eigenvectors of $L$ (or $L_{\text{sym}}$):

1. Compute $U = [u_2, u_3, \ldots, u_{k+1}] \in \mathbb{R}^{n \times k}$ (k Fiedler-like vectors)
2. Treat each row of $U$ as a point in $\mathbb{R}^k$
3. Run k-means on these rows

**Why this works:** the spectral embedding separates communities in ℝ^k even when they are not linearly separable in the original space.

In [3]:
def spectral_clustering(adj, k):
    import numpy as np
    L, nodes = graph_laplacian(adj)
    vals, vecs = np.linalg.eigh(L)
    # Use k smallest non-trivial eigenvectors
    U = vecs[:, 1:k+1]   # shape (n, k)
    # Simple k-means
    import random
    rng = random.Random(42)
    n = len(nodes)
    # Initialize centroids randomly
    centers = U[rng.sample(range(n), k)]
    labels = np.zeros(n, dtype=int)
    for _ in range(50):
        # Assign
        dists = np.array([[np.linalg.norm(U[i]-centers[c]) for c in range(k)] for i in range(n)])
        labels = np.argmin(dists, axis=1)
        # Update
        new_centers = np.array([U[labels==c].mean(axis=0) if (labels==c).any() else centers[c] for c in range(k)])
        if np.allclose(centers, new_centers): break
        centers = new_centers
    return {nodes[i]: labels[i] for i in range(n)}

part = spectral_clustering(adj, k=2)
true_part = {n:0 for n in c1}; true_part.update({n:1 for n in c2})
correct = sum(part[n]==true_part[n] or part[n]==(1-true_part[n]) for n in nodes)
print(f"Spectral clustering accuracy: {correct}/{len(nodes)}")
print("Partition:", {n:int(v) for n,v in part.items()})

Spectral clustering accuracy: 10/10
Partition: {0: 1, 1: 0, 2: 1, 3: 1, 4: 0, 5: 1, 6: 0, 7: 1, 8: 1, 9: 1}


## Summary

| Laplacian | Cuts minimized | Best for |
|-----------|---------------|---------|
| $L = D-A$ | RatioCut | Uniform degree graphs |
| $L_{sym}=D^{-1/2}LD^{-1/2}$ | NCut | Heterogeneous degree graphs |
| $L_{rw}=D^{-1}L$ | NCut (same) | Random walk interpretation |

Spectral clustering is more principled than k-means (doesn't assume spherical clusters) but is $O(n^3)$ for exact eigenvector computation. For large graphs, approximate eigenvectors via **Lanczos iteration** or **power method** are used.